In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score,mean_squared_error
from sklearn.linear_model import LinearRegression,SGDRegressor

In [ ]:
from sklearn.datasets import load_diabetes

In [ ]:
df = load_diabetes()
x = df.data
y = df.target


In [ ]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size = 0.2,random_state = 42)

In [ ]:
lr = LinearRegression(fit_intercept = True)
lr.fit(x_train,y_train)
y_pred = lr.predict(x_test)
print('r2_score:',r2_score(y_test,y_pred))
print(lr.coef_)
print(lr.intercept_)

In [ ]:
print('MSE',mean_squared_error(y_test,y_pred))

In [ ]:
sd = SGDRegressor(learning_rate = 'constant',max_iter = 150)
sd.fit(x_train,y_train)
y_pred = sd.predict(x_test)
print('r2_score:',r2_score(y_test,y_pred))
print(lr.coef_)
print(lr.intercept_)

GRADIENT DESCENT FROM SCRATCH

BATCH GRADIENT DESCENT

In [ ]:
import numpy as np

class LinearRegressionGD:
    def __init__(self, learning_rate=0.01, epochs=300, batch_size=None, verbose=False):
        self.lr = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size
        self.verbose = verbose

        self.weights = None
        self.bias = 0
        self.loss_history = []

   
    def scaler_fit(self, X):
        self.mean_ = np.mean(X, axis=0)
        self.std_ = np.std(X, axis=0)

        
        self.std_[self.std_ == 0] = 1

    def transform(self, X):
        return (X - self.mean_) / self.std_
    def _init_params(self, n_features):
        self.weights = np.zeros((n_features, 1))
        self.bias = 0
    def _loss(self, y, y_pred):
        return np.mean((y - y_pred) ** 2)

    
    def _grads(self, X, y, y_pred):
        n = len(y)
        error = y_pred - y

        dw = (2 / n) * (X.T @ error)
        db = (2 / n) * np.sum(error)

        return dw, db

   
    def fit(self, X, y):
        X = np.array(X)
        y = np.array(y).reshape(-1, 1)
        self.scaler_fit(X)
        X = self.transform(X)

        n_samples, n_features = X.shape
        self._init_params(n_features)

        for epoch in range(self.epochs):

            if self.batch_size:
                indices = np.random.permutation(n_samples)
                X_shuffled = X[indices]
                y_shuffled = y[indices]

                for i in range(0, n_samples, self.batch_size):
                    X_batch = X_shuffled[i:i+self.batch_size]
                    y_batch = y_shuffled[i:i+self.batch_size]

                    y_pred = X_batch @ self.weights + self.bias
                    dw, db = self._grads(X_batch, y_batch, y_pred)

                    self.weights -= self.lr * dw
                    self.bias -= self.lr * db

            else:
                y_pred = X @ self.weights + self.bias
                dw, db = self._grads(X, y, y_pred)

                self.weights -= self.lr * dw
                self.bias -= self.lr * db
            if epoch % 10 == 0:
                y_pred_full = X @ self.weights + self.bias
                loss = self._loss(y, y_pred_full)

                self.loss_history.append(loss)

                if self.verbose:
                    print(f"Epoch {epoch} | Loss: {loss:.5f}")

    
    def predict(self, X):
        X = np.array(X)
        X = self.transform(X)
        return X @ self.weights + self.bias

In [ ]:
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt


X, y = make_regression(n_samples=1000, n_features=1, noise=10, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LinearRegressionGD(
    learning_rate=0.01,
    epochs=1000,
    verbose=True
)

model.fit(X_train, y_train)


y_pred = model.predict(X_test).ravel()
y_test = y_test.ravel()


print("r2_score:", r2_score(y_test, y_pred))


plt.plot(model.loss_history)
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Loss Convergence")
plt.show()